In [2]:
import pandas as pd
import joblib
import logging
import warnings

# Suppress AIF360 dependency warnings and general warnings
logging.getLogger('root').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

from aif360.datasets import StandardDataset
from aif360.metrics import ClassificationMetric

# 1. Load trained model, encoders, and dataset
model = joblib.load('rf_model.joblib')
le_source = joblib.load('le_source.joblib')
le_pos = joblib.load('le_pos.joblib')
df_audit = pd.read_csv('HR_anonymized.csv')

# Reconstruct engineered features and encoded variables
# Calculating Tenure (Years) based on reference date
df_audit['DateofHire'] = pd.to_datetime(df_audit['DateofHire'])
ref_date = pd.to_datetime('2020-01-01')
df_audit['Tenure'] = (ref_date - df_audit['DateofHire']).dt.days / 365.25

# Applying pre-trained LabelEncoders to ensure data consistency
df_audit['RecruitmentSource_Enc'] = le_source.transform(df_audit['RecruitmentSource'])
df_audit['Position_Enc'] = le_pos.transform(df_audit['Position'])

# Defining features expected by the model
features = [
    'Salary', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 
    'Absences', 'DaysLateLast30', 'DeptID', 'PerfScoreID',
    'Tenure', 'RecruitmentSource_Enc', 'Position_Enc'
]

# Generating model predictions for the entire dataset
df_audit['Prediction'] = model.predict(df_audit[features])

print("--- FAIRNESS AUDIT REPORT (AIF360) ---")

# Filtering relevant columns for the audit to prevent format errors
cols_audit = features + ['GenderID', 'Prediction']
df_audit_clean = df_audit[cols_audit]

# Converting to AIF360 StandardDataset format
# GenderID: 1 = Male (Privileged), 0 = Female (Unprivileged)
dataset = StandardDataset(
    df=df_audit_clean,
    label_name='Prediction', 
    favorable_classes=[0],           # 0 indicates "Stay" (Positive outcome)
    protected_attribute_names=['GenderID'],
    privileged_classes=[[1]]         # Male as the reference group
)

# Defining groups for metric computation
privileged_groups = [{'GenderID': 1}]
unprivileged_groups = [{'GenderID': 0}]

# Computing fairness metrics
metric = ClassificationMetric(
    dataset, dataset, 
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

di = metric.disparate_impact()

print(f"Metric: Disparate Impact (DI)")
print(f"Calculated Value: {round(di, 3)}")

# Compliance check based on the "80% Rule" (Four-Fifths Rule)
if 0.8 < di < 1.25:
    print("Status: COMPLIANT. No significant bias detected against the protected attribute.")
else:
    print("Status: WARNING. Potential algorithmic bias detected (80% rule violation).")

print("\n--- Descriptive Analysis: Predicted Turnover Rate by Gender ---")
# Calculating the mean of 'Prediction' (Turnover probability per group)
risk_by_gender = df_audit.groupby('GenderID')['Prediction'].mean()
print(risk_by_gender)

--- FAIRNESS AUDIT REPORT (AIF360) ---
Metric: Disparate Impact (DI)
Calculated Value: 0.995
Status: COMPLIANT. No significant bias detected against the protected attribute.

--- Descriptive Analysis: Predicted Turnover Rate by Gender ---
GenderID
0    0.380682
1    0.377778
Name: Prediction, dtype: float64
